# Task 054: refnx — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: X-ray reflectometry inversion for thin film structure using refnx

**Paper**: References TFM and MSM papers but no specific pyTFM paper found
**Repository**: https://github.com/fabrylab/pyTFM

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 85.32 dB
- **SSIM**: N/A (1D reflectivity curve)
- **Evaluation**: X-ray reflectometry thin-film fitting (correlation_logR=0.9999998, thickness_error=0.075 Å)

### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
refnx_xrr — X-ray Reflectometry Inverse Problem
==================================================
Task: Recover thin-film layer structure (thickness, roughness, SLD)
      from measured X-ray reflectivity curve R(Q).

Inverse Problem:
    Given a reflectivity curve R(Q), recover the layer parameters
    (thickness d, roughness σ, SLD ρ) of a multilayer thin film.

Forward Model (Parratt/Abeles recursion via refnx):
    R(Q) = |r(Q)|²  where r is computed by refnx.reflect using
    the Abeles transfer-matrix formalism with Névot-Croce roughness.

Inverse Solver:
    Differential Evolution + Levenberg-Marquardt refinement
    using refnx.analysis.CurveFitter.

Repo: https://github.com/refnx/refnx
Paper: Nelson & Prescott (2019), J. Appl. Cryst., 52, 193.
       doi:10.1107/S1600576718017296

Metrics:
    - RMSE(log₁₀R): root-mean-square error in log₁₀(reflectivity) space
    - CC(log₁₀R): Pearson correlation of log₁₀(R) curves
    - Parameter errors: |Δd|, |ΔSLD|, |Δσ| for each fitted layer

Usage:
    /data/yjh/spectro_env/bin/python refnx_code.py
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`build_structure`, `sld_profile_from_params`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 2. Build refnx Structure (used for both forward & inverse)
# ═══════════════════════════════════════════════════════════
def build_structure(params, vary=False):
    """
    Build a refnx Structure from a parameter dict.

    Parameters
    ----------
    params : dict
        Keys: polymer_thick, polymer_sld, polymer_rough,
              sio2_thick, sio2_sld, sio2_rough,
              si_sld, si_rough, bkg
    vary : bool
        If True, set bounds and mark parameters as variable
        (for fitting). If False, fix all parameters (for
        forward generation).

    Returns
    -------
    structure : refnx.reflect.Structure
    model : refnx.reflect.ReflectModel
    """
    air = SLDobj(0.0, name='air')
    polymer = SLDobj(params["polymer_sld"], name='polymer')
    sio2 = SLDobj(params["sio2_sld"], name='sio2')
    si = SLDobj(params["si_sld"], name='si')

    # air → polymer
    polymer_slab = polymer(params["polymer_thick"], params["polymer_rough"])
    # polymer → SiO2
    sio2_slab = sio2(params["sio2_thick"], params["sio2_rough"])
    # SiO2 → Si substrate
    si_slab = si(0, params["si_rough"])

    if vary:
        polymer_slab.thick.setp(bounds=(50, 500), vary=True)
        polymer_slab.sld.real.setp(bounds=(0.3, 6.0), vary=True)
        polymer_slab.rough.setp(bounds=(1, 25), vary=True)
        sio2_slab.thick.setp(bounds=(3, 50), vary=True)
        sio2_slab.rough.setp(bounds=(0.5, 15), vary=True)
        si_slab.rough.setp(bounds=(0.5, 15), vary=True)

    structure = air | polymer_slab | sio2_slab | si_slab
    model = ReflectModel(structure, bkg=params["bkg"], dq=DQ_OVER_Q * 100)

    if vary:
        model.bkg.setp(bounds=(1e-10, 1e-5), vary=True)

    return structure, model, {
        "polymer_slab": polymer_slab,
        "sio2_slab": sio2_slab,
        "si_slab": si_slab,
    }

# ═══════════════════════════════════════════════════════════
# 7. Visualization
# ═══════════════════════════════════════════════════════════
def sld_profile_from_params(params, z_max=400, n_pts=500):
    """Compute SLD-vs-depth profile using refnx Structure."""
    structure, _, _ = build_structure(params, vary=False)
    z = np.linspace(0, z_max, n_pts)
    sld_profile = structure.sld_profile(z)
    return sld_profile[:, 0], sld_profile[:, 1]

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `forward_operator`, `load_or_generate_data`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 3. Forward Operator (using refnx ReflectModel)
# ═══════════════════════════════════════════════════════════
def forward_operator(params, q):
    """
    Compute specular X-ray reflectivity R(Q) using the refnx Abeles
    matrix formalism.

    This is the ACTUAL refnx forward engine — not a hand-coded
    reimplementation.  refnx internally uses the Parratt (or Abeles)
    recursive algorithm with Névot-Croce interfacial roughness factors
    and optional resolution smearing.

    Parameters
    ----------
    params : dict   Ground-truth layer parameters.
    q : np.ndarray  Momentum-transfer values [Å^-1].

    Returns
    -------
    R : np.ndarray  Reflectivity curve.
    """
    _, model, _ = build_structure(params, vary=False)
    R = model(q)
    return R

# ═══════════════════════════════════════════════════════════
# 4. Data Generation (Synthetic Benchmark)
# ═══════════════════════════════════════════════════════════
def load_or_generate_data():
    """
    Generate synthetic XRR benchmark data.

    1. Build ground-truth structure with refnx.
    2. Compute clean reflectivity R(Q) via refnx forward.
    3. Add realistic Poisson-like noise + background.
    """
    print("[DATA] Building ground-truth multilayer with refnx ...")
    q = np.linspace(Q_MIN, Q_MAX, N_POINTS)

    R_clean = forward_operator(GT_PARAMS, q)
    print(f"[DATA] R(Q) range: [{R_clean.min():.3e}, {R_clean.max():.3e}]")

    # Realistic noise: relative Gaussian noise scaled by sqrt(R)
    rng = np.random.default_rng(SEED)
    sigma_R = np.maximum(R_clean * NOISE_FRAC, 1e-12)
    R_noisy = R_clean + sigma_R * rng.standard_normal(N_POINTS)
    R_noisy = np.maximum(R_noisy, 1e-12)  # reflectivity ≥ 0

    # Error bars for the fitter
    R_err = sigma_R

    print(f"[DATA] Added {NOISE_FRAC*100:.0f}% relative noise "
          f"({N_POINTS} points, Q ∈ [{Q_MIN}, {Q_MAX}] Å⁻¹)")

    return q, R_noisy, R_clean, R_err

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `reconstruct`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 5. Inverse Solver (refnx CurveFitter)
# ═══════════════════════════════════════════════════════════
def reconstruct(q, R_meas, R_err):
    """
    Fit the XRR curve using refnx's Differential Evolution +
    Levenberg-Marquardt pipeline.

    The refnx CurveFitter wraps scipy.optimize.differential_evolution
    for global search, then scipy.optimize.least_squares for local
    refinement.  Resolution smearing (dq) is handled internally.

    Parameters
    ----------
    q : np.ndarray      Q values [Å^-1].
    R_meas : np.ndarray Measured reflectivity.
    R_err : np.ndarray  Error bars on reflectivity.

    Returns
    -------
    fitted_params : dict  Recovered layer parameters.
    R_fit : np.ndarray    Best-fit reflectivity curve.
    """
    # Build fitting model with free parameters
    initial_guess = {
        "polymer_thick": 200.0,   # deliberately off from GT
        "polymer_sld": 2.0,
        "polymer_rough": 5.0,
        "sio2_thick": 10.0,
        "sio2_sld": GT_PARAMS["sio2_sld"],  # fixed (known)
        "sio2_rough": 5.0,
        "si_sld": GT_PARAMS["si_sld"],       # fixed (known)
        "si_rough": 5.0,
        "bkg": 1e-7,
    }

    structure, model, slabs = build_structure(initial_guess, vary=True)

    # Create refnx dataset  (Q, R, dR, dQ)
    dq = q * DQ_OVER_Q
    dataset = ReflectDataset(data=(q, R_meas, R_err, dq))

    objective = Objective(model, dataset)

    # ── Stage 1: Differential Evolution (global) ──
    print("[RECON] Stage 1 — Differential Evolution (refnx.CurveFitter) ...")
    fitter = CurveFitter(objective)
    fitter.fit('differential_evolution', seed=SEED, maxiter=150, tol=1e-5)
    chi2_de = objective.chisqr()
    print(f"[RECON]   χ² after DE = {chi2_de:.4f}")

    # ── Stage 2: Least-Squares refinement (local) ──
    print("[RECON] Stage 2 — Least-Squares refinement ...")
    fitter.fit('least_squares')
    chi2_lm = objective.chisqr()
    print(f"[RECON]   χ² after LM = {chi2_lm:.4f}")

    # Extract fitted parameter values
    p = slabs["polymer_slab"]
    s = slabs["sio2_slab"]
    si = slabs["si_slab"]

    fitted_params = {
        "polymer_thick": float(p.thick.value),
        "polymer_sld": float(p.sld.real.value),
        "polymer_rough": float(p.rough.value),
        "sio2_thick": float(s.thick.value),
        "sio2_sld": float(s.sld.real.value),
        "sio2_rough": float(s.rough.value),
        "si_sld": GT_PARAMS["si_sld"],
        "si_rough": float(si.rough.value),
        "bkg": float(model.bkg.value),
    }

    R_fit = model(q, x_err=dq)

    return fitted_params, R_fit, chi2_lm

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`, `visualize_results`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 6. Evaluation Metrics
# ═══════════════════════════════════════════════════════════
def compute_metrics(gt_params, fit_params, R_clean, R_fit, q):
    """
    Compute reconstruction quality metrics.

    Reflectivity-space:
      - RMSE(log₁₀R)
      - CC(log₁₀R): Pearson correlation in log₁₀ space
      - PSNR(log₁₀R)
      - SSIM(log₁₀R)

    Parameter-space:
      - Absolute errors for thickness, SLD, roughness of each layer
      - Relative errors
    """
    from skimage.metrics import structural_similarity as ssim_fn

    logR_gt = np.log10(np.maximum(R_clean, 1e-12))
    logR_fit = np.log10(np.maximum(R_fit, 1e-12))

    # Reflectivity metrics
    rmse_logR = float(np.sqrt(np.mean((logR_gt - logR_fit) ** 2)))
    cc_logR = float(np.corrcoef(logR_gt, logR_fit)[0, 1])

    data_range = logR_gt.max() - logR_gt.min()
    mse = np.mean((logR_gt - logR_fit) ** 2)
    psnr_logR = float(10 * np.log10(data_range ** 2 / max(mse, 1e-30)))

    # 1-D SSIM: tile to make 2D (7×N) so win_size=7 works
    tile_rows = 7
    a2d = np.tile(logR_gt, (tile_rows, 1))
    b2d = np.tile(logR_fit, (tile_rows, 1))
    ssim_logR = float(ssim_fn(
        a2d, b2d,
        data_range=data_range, win_size=7
    ))

    # Parameter recovery
    param_keys = [
        ("polymer_thick", "d_polymer [Å]"),
        ("polymer_sld", "SLD_polymer [1e-6 Å⁻²]"),
        ("polymer_rough", "σ_polymer [Å]"),
        ("sio2_thick", "d_SiO₂ [Å]"),
        ("sio2_rough", "σ_SiO₂ [Å]"),
        ("si_rough", "σ_Si [Å]"),
    ]

    param_errors = {}
    for key, label in param_keys:
        gt = gt_params[key]
        fit = fit_params[key]
        err = abs(gt - fit)
        rel = err / max(abs(gt), 1e-12) * 100
        param_errors[f"gt_{key}"] = float(gt)
        param_errors[f"fit_{key}"] = float(fit)
        param_errors[f"abs_err_{key}"] = float(err)
        param_errors[f"rel_err_{key}"] = float(rel)

    metrics = {
        "PSNR_logR": psnr_logR,
        "SSIM_logR": ssim_logR,
        "CC_logR": cc_logR,
        "RMSE_logR": rmse_logR,
        **param_errors,
    }
    return metrics

def visualize_results(q, R_meas, R_clean, R_fit,
                      gt_params, fit_params, metrics, save_path):
    """
    Three-panel figure:
      1. Reflectivity curves (log scale) with residuals
      2. SLD-vs-depth profile: GT vs fit
      3. Bar chart of parameter recovery
    """
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    # (a) Reflectivity
    ax = axes[0, 0]
    ax.semilogy(q, R_clean, 'b-', lw=2, label='Ground Truth', zorder=3)
    ax.semilogy(q, R_meas, 'k.', ms=1.5, alpha=0.4, label='Noisy Data')
    ax.semilogy(q, R_fit, 'r--', lw=1.5, label='refnx Fit', zorder=2)
    ax.set_xlabel('Q [Å⁻¹]')
    ax.set_ylabel('Reflectivity R(Q)')
    ax.set_title('(a) X-ray Reflectivity')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # (b) Residuals in log space
    ax = axes[0, 1]
    logR_gt = np.log10(np.maximum(R_clean, 1e-12))
    logR_fit = np.log10(np.maximum(R_fit, 1e-12))
    ax.plot(q, logR_gt - logR_fit, 'g-', lw=1)
    ax.axhline(0, color='k', ls='--', lw=0.5)
    ax.set_xlabel('Q [Å⁻¹]')
    ax.set_ylabel('Δ log₁₀R')
    ax.set_title(f'(b) Residuals  RMSE={metrics["RMSE_logR"]:.4f}')
    ax.grid(True, alpha=0.3)

    # (c) SLD profile
    ax = axes[1, 0]
    try:
        z_gt, sld_gt = sld_profile_from_params(gt_params)
        z_fit, sld_fit = sld_profile_from_params(fit_params)
        ax.plot(z_gt, sld_gt, 'b-', lw=2, label='Ground Truth')
        ax.plot(z_fit, sld_fit, 'r--', lw=2, label='refnx Fit')
    except Exception:
        # Fallback: simple step plot
        ax.text(0.5, 0.5, 'SLD profile unavailable',
                transform=ax.transAxes, ha='center')
    ax.set_xlabel('Depth [Å]')
    ax.set_ylabel('SLD [10⁻⁶ Å⁻²]')
    ax.set_title('(c) SLD Profile')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # (d) Parameter bar chart
    ax = axes[1, 1]
    labels = ['d_poly', 'SLD_poly', 'σ_poly', 'd_SiO₂', 'σ_SiO₂', 'σ_Si']
    keys = ['polymer_thick', 'polymer_sld', 'polymer_rough',
            'sio2_thick', 'sio2_rough', 'si_rough']
    gt_vals = [gt_params[k] for k in keys]
    fit_vals = [fit_params[k] for k in keys]
    x = np.arange(len(labels))
    w = 0.35
    ax.bar(x - w/2, gt_vals, w, label='GT', color='steelblue')
    ax.bar(x + w/2, fit_vals, w, label='Fit', color='tomato')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=9, rotation=15)
    ax.set_title('(d) Parameter Recovery')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

    fig.suptitle(
        f"refnx — X-ray Reflectometry Inversion\n"
        f"PSNR(logR)={metrics['PSNR_logR']:.1f} dB  |  "
        f"SSIM(logR)={metrics['SSIM_logR']:.4f}  |  "
        f"CC(logR)={metrics['CC_logR']:.4f}",
        fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[VIS] Saved → {save_path}")

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
print("=" * 65)
print("  refnx — X-ray Reflectometry Inverse Problem")
print("=" * 65)

### Step 1: Generate data with refnx forward model

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# Step 1: Generate data with refnx forward model
q, R_meas, R_clean, R_err = load_or_generate_data()

### Step 2: Inverse with refnx DE + LM

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Step 2: Inverse with refnx DE + LM
print("\n[RECON] Fitting with refnx.analysis.CurveFitter ...")
fit_params, R_fit, chi2 = reconstruct(q, R_meas, R_err)

### Step 3: Metrics

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# Step 3: Metrics
print("\n[EVAL] Computing metrics ...")
metrics = compute_metrics(GT_PARAMS, fit_params, R_clean, R_fit, q)
for k, v in sorted(metrics.items()):
    print(f"  {k:30s} = {v}")

### Step 4: Save

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# Step 4: Save
with open(os.path.join(RESULTS_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
np.save(os.path.join(RESULTS_DIR, "reconstruction.npy"), R_fit)
np.save(os.path.join(RESULTS_DIR, "ground_truth.npy"), R_clean)
np.save(os.path.join(RESULTS_DIR, "measurements.npy"), R_meas)

### Step 5: Visualize

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Step 5: Visualize
visualize_results(q, R_meas, R_clean, R_fit,
                  GT_PARAMS, fit_params, metrics,
                  os.path.join(RESULTS_DIR, "reconstruction_result.png"))

print("\n" + "=" * 65)
print("  DONE — refnx XRR benchmark complete")
print("=" * 65)

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **refnx**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=85.32 dB, SSIM=N/A (1D reflectivity curve))

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: References TFM and MSM papers but no specific pyTFM paper found
- Repository: https://github.com/fabrylab/pyTFM
